# ============================================
#  Notebook 04b — Per-Case Text Consolidation
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 04b: Per-Case Text Consolidation

**Purpose:**
Bridge between NB04 (per-document text extraction) and NB05 (BERT/feature extraction).

NB04 outputs one `.txt` per **individual PDF** (document-level `case_id` = hash of filename).  
This notebook groups those files by **patient case folder** and consolidates all documents  
for each patient into a single text file with section headers.

**Inputs (from NB04):**
- `DATA_PRIVATE_DIR/deidentified/case_document_mapping.csv` — maps document case_id → original path
- `DATA_PRIVATE_DIR/extracted_text/*.txt` — one file per document

**Outputs:**
- `DATA_PRIVATE_DIR/extracted_text_consolidated/{patient_case_id}.txt` — one file per patient case
- `DATA_PRIVATE_DIR/deidentified/patient_case_manifest.csv` — case-level metadata
- `data/processed/patient_case_manifest.csv` — non-PHI version (no paths) committed to repo
- `reports/consolidation_summary.png` — document count and type distribution

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv("PROJECT_ROOT",
    "/Users/robertjames/Documents/GitHub/llm_summarization_br_ca"))
DATA_PRIVATE_DIR = Path(os.getenv("DATA_PRIVATE_DIR", "/Users/robertjames/data_private"))

DEID_DIR         = DATA_PRIVATE_DIR / "deidentified"
TEXT_DIR         = DATA_PRIVATE_DIR / "extracted_text"
CONSOLIDATED_DIR = DATA_PRIVATE_DIR / "extracted_text_consolidated"
PROCESSED_DIR    = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR       = PROJECT_ROOT / "reports"

CONSOLIDATED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_PATH = DEID_DIR / "case_document_mapping.csv"

print(f"TEXT_DIR         : {TEXT_DIR}")
print(f"CONSOLIDATED_DIR : {CONSOLIDATED_DIR}")
print(f"Mapping CSV      : {MAPPING_PATH}")
print(f"Mapping exists   : {MAPPING_PATH.exists()}")

---
## Part 1: Load Document Mapping and Parse Patient Case IDs

The `original_path` from NB04 follows the structure:
```
DATA_PRIVATE_DIR/raw/SurgeonName/SurgeonInitials_PatientInitials_CaseType/document.pdf
```
The **patient case ID** is the immediate parent folder of the PDF (e.g., `AB_CB_invasive`).
The **surgeon name** is the grandparent folder.

In [ ]:
if not MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"case_document_mapping.csv not found at {MAPPING_PATH}.\n"
        "Run NB04 first to generate this file."
    )

mapping_df = pd.read_csv(MAPPING_PATH)
print(f"Document mapping loaded: {mapping_df.shape[0]} documents")
print(f"Columns: {list(mapping_df.columns)}")
mapping_df.head(3)

In [ ]:
def parse_patient_case_id(original_path: str) -> str:
    """Extract the patient case folder name from the original PDF path.
    e.g. '.../Andrea Barrio/AB_CB_invasive/imaging_mammo.pdf' -> 'AB_CB_invasive'
    """
    return Path(original_path).parent.name

def parse_surgeon_name(original_path: str) -> str:
    """Extract the surgeon folder name from the original PDF path.
    e.g. '.../Andrea Barrio/AB_CB_invasive/imaging_mammo.pdf' -> 'Andrea Barrio'
    """
    return Path(original_path).parent.parent.name

def parse_case_type(patient_case_id: str) -> str:
    """Infer invasive vs DCIS from case folder name suffix."""
    pid_lower = patient_case_id.lower()
    if "dcis" in pid_lower:
        return "DCIS"
    elif "invasive" in pid_lower:
        return "invasive"
    return "unknown"

mapping_df["patient_case_id"] = mapping_df["original_path"].apply(parse_patient_case_id)
mapping_df["surgeon_name"]    = mapping_df["original_path"].apply(parse_surgeon_name)
mapping_df["case_type"]       = mapping_df["patient_case_id"].apply(parse_case_type)

n_cases = mapping_df["patient_case_id"].nunique()
print(f"Unique patient cases: {n_cases}")
print(f"Case type breakdown:")
print(mapping_df["case_type"].value_counts().to_string())
mapping_df[["patient_case_id", "surgeon_name", "case_type", "doc_type_inferred"]].head(6)

---
## Part 2: Document Type Classification

Within each patient case, documents are ordered and labelled:

| Section | Keywords in filename |
|---|---|
| HPI / Clinical Notes | `hpi`, `summary`, `note` |
| Radiology Reports | `imaging`, `mammo`, `mri`, `us`, `ultrasound`, `rad` |
| Pathology Reports | `path`, `biopsy`, `histol`, `receptor`, `surgical` |
| Genetics | `genetic` |
| Other | anything else |

In [ ]:
SECTION_ORDER = ["hpi", "radiology", "pathology", "genetics", "unknown"]

SECTION_HEADERS = {
    "hpi":       "=== HPI / CLINICAL NOTES ===",
    "radiology": "=== RADIOLOGY REPORTS ===",
    "pathology": "=== PATHOLOGY REPORTS ===",
    "genetics":  "=== GENETICS / MOLECULAR ===",
    "unknown":   "=== OTHER DOCUMENTS ===",
}

def classify_doc_section(filename: str) -> str:
    fn = filename.lower()
    if any(kw in fn for kw in ["hpi", "summary", "note"]):
        return "hpi"
    elif any(kw in fn for kw in ["imaging", "mammo", "mri", "_us_", "_us2",
                                   "ultrasound", "rad", "bs_", "ctap", "ct_chest"]):
        return "radiology"
    elif any(kw in fn for kw in ["path", "biopsy", "bopsy", "histol",
                                   "receptor", "surgical"]):
        return "pathology"
    elif any(kw in fn for kw in ["genetic", "genetics"]):
        return "genetics"
    return "unknown"

mapping_df["section"] = mapping_df["original_filename"].apply(classify_doc_section)

print("Document section distribution:")
print(mapping_df["section"].value_counts().to_string())
mapping_df[["original_filename", "patient_case_id", "section"]].head(8)

---
## Part 3: Consolidate Documents Per Patient Case

For each patient case:
1. Load each individual `.txt` file (keyed by `case_id` = hash of filename)
2. Sort documents by section order (HPI → Radiology → Pathology → Genetics → Other)
3. Write consolidated text with section headers and document sub-headers

In [ ]:
def load_doc_text(case_id: str, text_dir: Path) -> str:
    txt_path = text_dir / f"{case_id}.txt"
    if txt_path.exists():
        return txt_path.read_text(encoding="utf-8").strip()
    return ""

def consolidate_case(
    case_df: pd.DataFrame,
    text_dir: Path,
    section_order: list = SECTION_ORDER,
    section_headers: dict = SECTION_HEADERS,
) -> str:
    """
    Given all document rows for one patient case, build a single consolidated
    text string with section headers and per-document sub-headers.
    """
    blocks = []
    for section in section_order:
        docs = case_df[case_df["section"] == section].copy()
        if docs.empty:
            continue
        blocks.append(section_headers[section])
        blocks.append("")
        for _, row in docs.iterrows():
            text = load_doc_text(row["case_id"], text_dir)
            if not text:
                continue
            sub_header = f"--- {row['original_filename']} ---"
            blocks.append(sub_header)
            blocks.append(text)
            blocks.append("")
    return "\n".join(blocks).strip()

# Run consolidation
manifest_rows = []
n_written = 0
n_empty   = 0

for patient_case_id, case_df in mapping_df.groupby("patient_case_id"):
    consolidated_text = consolidate_case(case_df, TEXT_DIR)

    if not consolidated_text.strip():
        n_empty += 1
        status = "EMPTY"
    else:
        out_path = CONSOLIDATED_DIR / f"{patient_case_id}.txt"
        out_path.write_text(consolidated_text, encoding="utf-8")
        n_written += 1
        status = "OK"

    manifest_rows.append({
        "patient_case_id": patient_case_id,
        "surgeon_name":    case_df["surgeon_name"].iloc[0],
        "case_type":       case_df["case_type"].iloc[0],
        "n_docs_total":    len(case_df),
        "n_hpi":           (case_df["section"] == "hpi").sum(),
        "n_radiology":     (case_df["section"] == "radiology").sum(),
        "n_pathology":     (case_df["section"] == "pathology").sum(),
        "n_genetics":      (case_df["section"] == "genetics").sum(),
        "n_unknown":       (case_df["section"] == "unknown").sum(),
        "consolidated_chars": len(consolidated_text),
        "status":          status,
    })

df_manifest = pd.DataFrame(manifest_rows)

print(f"Cases written    : {n_written}")
print(f"Cases empty      : {n_empty}  (no extracted text found — run NB04 first)")
print(f"\nConsolidated files → {CONSOLIDATED_DIR}")
df_manifest.head(5)

---
## Part 4: Save Manifests

In [ ]:
# Full manifest (with surgeon names) — private, not committed
df_manifest.to_csv(DEID_DIR / "patient_case_manifest.csv", index=False)
print(f"Private manifest → {DEID_DIR / 'patient_case_manifest.csv'}")

# Non-PHI version — drop surgeon name, committed to repo
non_phi_cols = ["patient_case_id", "case_type", "n_docs_total",
                "n_hpi", "n_radiology", "n_pathology", "n_genetics",
                "n_unknown", "consolidated_chars", "status"]
df_manifest[non_phi_cols].to_csv(
    PROCESSED_DIR / "patient_case_manifest.csv", index=False
)
print(f"Non-PHI manifest → {PROCESSED_DIR / 'patient_case_manifest.csv'}")

# Also update the full document mapping with parsed columns
mapping_df.to_csv(DEID_DIR / "case_document_mapping.csv", index=False)
print(f"Updated mapping  → {DEID_DIR / 'case_document_mapping.csv'}")

---
## Part 5: Summary Statistics and Visualizations

In [ ]:
ok = df_manifest[df_manifest["status"] == "OK"]

print(f"Total patient cases: {len(df_manifest)}")
print(f"  OK (text available): {len(ok)}")
print(f"  EMPTY (NB04 not yet run): {(df_manifest['status']=='EMPTY').sum()}")
print()
if len(ok) > 0:
    print(f"Case type breakdown:")
    print(ok["case_type"].value_counts().to_string())
    print()
    print(f"Documents per case (mean ± std):")
    print(f"  Total  : {ok['n_docs_total'].mean():.1f} ± {ok['n_docs_total'].std():.1f}")
    print(f"  Radiology : {ok['n_radiology'].mean():.1f} ± {ok['n_radiology'].std():.1f}")
    print(f"  Pathology : {ok['n_pathology'].mean():.1f} ± {ok['n_pathology'].std():.1f}")
    print(f"  HPI       : {ok['n_hpi'].mean():.1f} ± {ok['n_hpi'].std():.1f}")
    print()
    print(f"Consolidated text length (chars):")
    print(ok["consolidated_chars"].describe().round(0).to_string())

In [ ]:
if len(ok) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Per-Case Text Consolidation Summary", fontsize=16, fontweight="bold")

    # 1. Docs per case histogram
    axes[0, 0].hist(ok["n_docs_total"], bins=range(1, ok["n_docs_total"].max() + 2),
                    color="#3498db", edgecolor="black", align="left")
    axes[0, 0].set_xlabel("Total Documents per Case")
    axes[0, 0].set_ylabel("Number of Cases")
    axes[0, 0].set_title("Documents per Patient Case")

    # 2. Document type breakdown stacked bar (mean per case)
    type_means = ok[["n_hpi", "n_radiology", "n_pathology", "n_genetics"]].mean()
    colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]
    axes[0, 1].bar(type_means.index, type_means.values, color=colors, edgecolor="black")
    axes[0, 1].set_ylabel("Mean Docs per Case")
    axes[0, 1].set_title("Mean Documents per Case by Type")
    axes[0, 1].tick_params(axis="x", rotation=15)

    # 3. Consolidated chars distribution
    axes[0, 2].hist(ok["consolidated_chars"] / 1000, bins=20,
                    color="#9b59b6", edgecolor="black")
    axes[0, 2].set_xlabel("Consolidated Text Length (thousands of chars)")
    axes[0, 2].set_title("Consolidated Text Length per Case")

    # 4. Invasive vs DCIS case type
    ct = ok["case_type"].value_counts()
    axes[1, 0].bar(ct.index, ct.values, color=["#e74c3c", "#3498db", "#95a5a6"],
                   edgecolor="black")
    axes[1, 0].set_ylabel("Number of Cases")
    axes[1, 0].set_title("Invasive vs DCIS Case Types")

    # 5. Cases per surgeon
    if "surgeon_name" in df_manifest.columns:
        surgeon_counts = ok["surgeon_name"].value_counts()
        axes[1, 1].barh(surgeon_counts.index, surgeon_counts.values,
                        color="#1abc9c", edgecolor="black")
        axes[1, 1].set_xlabel("Number of Cases")
        axes[1, 1].set_title("Cases per Surgeon")
        axes[1, 1].tick_params(axis="y", labelsize=8)

    # 6. Radiology vs Pathology doc count scatter
    axes[1, 2].scatter(ok["n_radiology"], ok["n_pathology"],
                       c=ok["case_type"].map({"invasive": "#e74c3c", "DCIS": "#3498db", "unknown": "grey"}),
                       s=60, alpha=0.7, edgecolors="black", linewidths=0.5)
    axes[1, 2].set_xlabel("N Radiology Docs")
    axes[1, 2].set_ylabel("N Pathology Docs")
    axes[1, 2].set_title("Radiology vs Pathology Doc Count")
    from matplotlib.patches import Patch
    axes[1, 2].legend(handles=[
        Patch(color="#e74c3c", label="Invasive"),
        Patch(color="#3498db", label="DCIS"),
    ], fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "consolidation_summary.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved → {OUTPUT_DIR / 'consolidation_summary.png'}")
else:
    print("No consolidated cases to plot — run NB04 first to extract text.")

---
## Part 6: Validation — Spot-Check Consolidated Text

In [ ]:
# Print the first 1500 chars of the first consolidated case for review
consolidated_files = sorted(CONSOLIDATED_DIR.glob("*.txt"))
if consolidated_files:
    sample_path = consolidated_files[0]
    sample_text = sample_path.read_text(encoding="utf-8")
    print(f"Sample case: {sample_path.name}")
    print(f"Total chars: {len(sample_text):,}")
    print()
    print(sample_text[:1500])
else:
    print("No consolidated files found — run NB04 first.")

In [ ]:
# Check that all expected cases are covered
expected_cases = set(mapping_df["patient_case_id"].unique())
written_cases  = {f.stem for f in CONSOLIDATED_DIR.glob("*.txt")}
missing        = expected_cases - written_cases

print(f"Expected cases : {len(expected_cases)}")
print(f"Written files  : {len(written_cases)}")
print(f"Missing        : {len(missing)}")
if missing:
    print("Cases with no consolidated file (NB04 text not yet extracted):")
    for m in sorted(missing):
        print(f"  {m}")

---
## Part 7: Feature-Aware Context Dictionary

Imports `feature_document_context.py` from `src/llm_eval_by_llm/` and validates that:
1. All 14 features have a defined source-document context
2. Each feature's `primary_sections` aligns with the section headers written in Part 3
3. `filter_text_to_relevant_sections()` correctly narrows the consolidated text per feature

This dictionary drives downstream LLM extraction (NB09) and prompt construction —
each feature only receives the document sections that can actually contain it.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from llm_eval_by_llm.feature_document_context import (
    FEATURE_DOCUMENT_CONTEXT,
    RADIOLOGY_FEATURES,
    PATHOLOGY_FEATURES,
    HPI_FEATURES,
    MIXED_FEATURES,
    HIGH_FABRICATION_RISK,
    FEATURE_SECTIONS,
    EXTRACTION_HINTS,
    filter_text_to_relevant_sections,
)

print(f"Features defined : {len(FEATURE_DOCUMENT_CONTEXT)}")
print()
print(f"{'Feature Key':<40} {'Domain':<12} {'Sections':<35} {'Fab Risk'}")
print("-" * 100)
for key, ctx in FEATURE_DOCUMENT_CONTEXT.items():
    sections_str = ", ".join(ctx["primary_sections"])
    print(f"{key:<40} {ctx['domain']:<12} {sections_str:<35} {ctx['fabrication_risk']}")

In [ ]:
# ── Validate section alignment ────────────────────────────────────────────────
# All sections referenced in the context dict must exist in SECTION_HEADERS (Part 3)
valid_sections = set(SECTION_HEADERS.keys())  # {"hpi", "radiology", "pathology", "genetics", "unknown"}

print("Section alignment check:")
all_ok = True
for key, ctx in FEATURE_DOCUMENT_CONTEXT.items():
    bad = [s for s in ctx["primary_sections"] if s not in valid_sections]
    if bad:
        print(f"  ✗ {key}: unknown section(s) {bad}")
        all_ok = False
if all_ok:
    print("  ✓ All 14 features reference valid section keys\n")

# ── Domain breakdown ──────────────────────────────────────────────────────────
print(f"Radiology-primary features  ({len(RADIOLOGY_FEATURES)}): {RADIOLOGY_FEATURES}")
print(f"Pathology-primary features  ({len(PATHOLOGY_FEATURES)}): {PATHOLOGY_FEATURES}")
print(f"HPI-primary features        ({len(HPI_FEATURES)}):       {HPI_FEATURES}")
print(f"Mixed-domain features       ({len(MIXED_FEATURES)}):     {MIXED_FEATURES}")
print()

# ── High fabrication-risk features ───────────────────────────────────────────
print(f"High fabrication-risk ({len(HIGH_FABRICATION_RISK)} features — require strict grounding):")
for k in HIGH_FABRICATION_RISK:
    print(f"  • {FEATURE_DOCUMENT_CONTEXT[k]['display']}")

In [ ]:
# ── Demo: filter consolidated text to the relevant sections per feature ────────
# Shows exactly what text would be passed to the LLM for each feature

if consolidated_files:
    sample_text = consolidated_files[0].read_text(encoding="utf-8")
    sample_case = consolidated_files[0].stem

    print(f"Case: {sample_case}  ({len(sample_text):,} total chars)\n")
    print("=" * 70)

    demo_features = [
        "lesion_size",              # radiology only
        "histologic_diagnosis",     # pathology only
        "chronology_preserved",     # hpi only
        "lymph_node",               # mixed (radiology + pathology + hpi)
        "additional_enhancement_mri",  # radiology only — high fabrication risk
        "receptor_status",          # pathology + genetics
    ]

    for feature_key in demo_features:
        ctx = FEATURE_DOCUMENT_CONTEXT[feature_key]
        filtered = filter_text_to_relevant_sections(
            sample_text, feature_key,
            section_header_map={k: v for k, v in SECTION_HEADERS.items()
                                 if k != "unknown"}
        )
        pct = len(filtered) / len(sample_text) * 100 if sample_text else 0
        print(f"\n[{ctx['fabrication_risk'].upper():6}] {ctx['display']}")
        print(f"  Sections : {ctx['primary_sections']}")
        print(f"  Context  : {len(filtered):,} chars  ({pct:.0f}% of full case text)")
        print(f"  Hint     : {ctx['extraction_hint'][:120]}...")

    print("\n" + "=" * 70)
    print("filter_text_to_relevant_sections() ready for NB09 prompt construction.")
else:
    print("No consolidated files yet — run NB04 first, then re-run this cell.")

---
## Summary of Outputs

| Output | Path | Committed? |
|--------|------|------------|
| Consolidated per-case `.txt` files | `DATA_PRIVATE_DIR/extracted_text_consolidated/` | No (PHI-adjacent) |
| Full manifest (with surgeon names) | `DATA_PRIVATE_DIR/deidentified/patient_case_manifest.csv` | No |
| Non-PHI manifest | `data/processed/patient_case_manifest.csv` | Yes |
| Consolidation plots | `reports/consolidation_summary.png` | Yes |

**Next step:** NB05 (`05_feature_extraction_ocr_bert.ipynb`) should point `TEXT_DIR` to  
`DATA_PRIVATE_DIR/extracted_text_consolidated/` for case-level BERT embeddings and text features.